# Exercise 5: Round trips

*Scripts and Encoding · version 0.1.0*

> Take 50 Uzbek sentences in Latin script (for example from FLORES-200, `uzn_Latn`). Convert them to Cyrillic and back with `Transliterator("uzb", …)`. Which sentences do not survive the round trip unchanged? Classify the differences (apostrophe variants for *oʻ*/*gʻ*, the tutuq belgisi, loanwords, capitalisation), and say which ones matter for a corpus.

FLORES-200 requires accepting its terms on Hugging Face; the notebook uses the first 50 Uzbek sentences of the openly available X-WMT test set instead.

**Setup.** `pip install "turkicnlp==0.5.0" requests`. Cells that fetch data from the web need internet access.

In [ ]:
import requests

BASE = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test"

def xwmt_side(pair, lang):
    """One side of an X-WMT test pair, e.g. xwmt_side("en-ky", "ky")."""
    r = requests.get(f"{BASE}/{pair}/{pair}.{lang}", timeout=30)
    r.raise_for_status()
    return r.text.splitlines()

In [ ]:
import re
from collections import Counter
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

uz = xwmt_side("en-uz", "uz")[:50]
to_cyr = Transliterator("uzb", Script.LATIN, Script.CYRILLIC)
to_lat = Transliterator("uzb", Script.CYRILLIC, Script.LATIN)
back = [to_lat.transliterate(to_cyr.transliterate(s)) for s in uz]

APOSTROPHES = "'`‘’ʻʼ´"
print("apostrophe characters in the input :", {f"U+{ord(c):04X}": n for c, n in Counter(c for s in uz for c in s if c in APOSTROPHES).items()})
print("apostrophe characters in the output:", {f"U+{ord(c):04X}": n for c, n in Counter(c for s in back for c in s if c in APOSTROPHES).items()})
print("sentences changed:", sum(a != b for a, b in zip(uz, back)), "of", len(uz))

In [ ]:
unify = lambda s: re.sub(f"[{APOSTROPHES}]", "'", s)

categories = Counter()
examples = {}
for a, b in zip(uz, back):
    for x, y in zip(a.split(), b.split()):
        if x == y:
            continue
        if unify(x) == unify(y):
            cat = "apostrophe variant only"
        elif x.lower() == y.lower():
            cat = "capitalisation"
        else:
            cat = "other (loanword, tutuq belgisi, ...)"
        categories[cat] += 1
        examples.setdefault(cat, (x, y))
print(dict(categories))
print(examples)

## Worked solution (TurkicNLP 0.5.0, run on 18 October 2026)

48 of the 50 sentences come back changed, but almost all changes are of one kind:

- **Apostrophe variant.** The X-WMT text writes *oʻ*, *gʻ* and the tutuq belgisi with the typographic apostrophe U+2019 (*qo’shimcha*, *ma’lumot*). The toolkit normalises every apostrophe variant to the ASCII apostrophe U+0027 before converting, so the round trip returns *qo'shimcha*, *ma'lumot*. The spelling is the same; the bytes differ.
- **Capitalisation.** One word: the abbreviation *AQSh* 'USA' comes back as *AQSH*, because Cyrillic *Ш* in an all-capital word is converted to all-capital *SH*.
- **Loanwords** (none in this sample): Russian loanwords such as *film* ↔ *фильм* would show up here, because the Latin form does not show the Cyrillic soft sign.

**What matters for a corpus.** The apostrophe changes split the vocabulary (*qo’shimcha* and *qo'shimcha* are different strings) unless the corpus fixes one convention and normalises to it; record the chosen code point. Capitalisation of abbreviations and loanword spellings change the actual text and need exception lists or a check against the original. A round trip that changes only apostrophes is lossless for the language but not for byte-level deduplication or exact-match evaluation.